In [2]:
# Importamos las librerías necesarias para generar los datos.
import os
from pathlib import Path

import pandas as pd
from faker import Faker
from dotenv import load_dotenv
from google.cloud import bigquery

In [3]:
# Localizamos la raíz del proyecto.
PROJECT_ROOT = Path.cwd().parents[1]

# Cargamos las variables del archivo .env.
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Leemos la configuración de Google Cloud.
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

# Construimos la ruta absoluta a las credenciales.
CREDENTIALS_PATH = PROJECT_ROOT / os.getenv(
    "GOOGLE_APPLICATION_CREDENTIALS"
)

# Configuramos las credenciales para Google Cloud.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(CREDENTIALS_PATH)

print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)
print("Credenciales:", CREDENTIALS_PATH)
print("¿Existe el archivo?:", CREDENTIALS_PATH.exists())

Proyecto: tc-sql-miguel
Dataset: electromarket
Credenciales: c:\Users\mgpir\Desktop\RepoPracticaObligatoria\bootcamp_AI_Engineering_05_26\tc-sql-lopezmiguel\credentials\service-account.json
¿Existe el archivo?: True


In [4]:
# Creamos el cliente de BigQuery.
client = bigquery.Client(project=PROJECT_ID)

print("Conexión con BigQuery correcta.")

Conexión con BigQuery correcta.


In [5]:
# Inicializamos Faker para generar datos en español.
fake = Faker("es_ES")

print("Faker inicializado correctamente.")

Faker inicializado correctamente.


In [6]:
# Definimos el volumen de datos que vamos a generar.
NUM_CUSTOMERS = 500
NUM_PRODUCTS = 70
NUM_ORDERS = 2000
NUM_ORDER_ITEMS = 4500

print("Clientes:", NUM_CUSTOMERS)
print("Productos:", NUM_PRODUCTS)
print("Pedidos:", NUM_ORDERS)
print("Líneas de pedido:", NUM_ORDER_ITEMS)

Clientes: 500
Productos: 70
Pedidos: 2000
Líneas de pedido: 4500


In [7]:
# Definimos las categorías de productos.
categories_data = [
    {
        "category_id": 1,
        "name": "Smartphones",
        "description": "Teléfonos inteligentes y dispositivos móviles."
    },
    {
        "category_id": 2,
        "name": "Laptops",
        "description": "Ordenadores portátiles para trabajo y uso personal."
    },
    {
        "category_id": 3,
        "name": "Audio",
        "description": "Auriculares, altavoces y dispositivos de sonido."
    },
    {
        "category_id": 4,
        "name": "Peripherals",
        "description": "Periféricos para ordenadores y estaciones de trabajo."
    },
    {
        "category_id": 5,
        "name": "Wearables",
        "description": "Relojes inteligentes y dispositivos tecnológicos portables."
    },
    {
        "category_id": 6,
        "name": "Accessories",
        "description": "Accesorios y complementos para dispositivos electrónicos."
    },
    {
        "category_id": 7,
        "name": "Gaming",
        "description": "Productos y accesorios para videojuegos."
    },
]

categories_df = pd.DataFrame(categories_data)

print(f"Categorías generadas: {len(categories_df)}")
categories_df

Categorías generadas: 7


,category_id,name,description
0,1,Smartphones,Teléfonos inteligentes y dispositivos móviles.
1,2,Laptops,Ordenadores portátiles para trabajo y uso pers...
2,3,Audio,"Auriculares, altavoces y dispositivos de sonido."
3,4,Peripherals,Periféricos para ordenadores y estaciones de t...
4,5,Wearables,Relojes inteligentes y dispositivos tecnológic...
5,6,Accessories,Accesorios y complementos para dispositivos el...
6,7,Gaming,Productos y accesorios para videojuegos.


In [8]:
# Comprobamos que las categorías tienen IDs únicos y no tienen valores nulos.
print("IDs únicos:", categories_df["category_id"].is_unique)
print("Valores nulos:", categories_df.isnull().sum().sum())

IDs únicos: True
Valores nulos: 0


In [9]:
# Definimos países europeos y algunas ciudades para generar clientes.
locations = {
    "Spain": ["Madrid", "Barcelona", "Valencia", "Seville", "Bilbao"],
    "France": ["Paris", "Lyon", "Marseille", "Toulouse"],
    "Germany": ["Berlin", "Munich", "Hamburg", "Frankfurt"],
    "Italy": ["Rome", "Milan", "Naples", "Turin"],
    "Portugal": ["Lisbon", "Porto", "Braga"],
    "Netherlands": ["Amsterdam", "Rotterdam", "Utrecht"],
    "Belgium": ["Brussels", "Antwerp", "Ghent"],
    "Ireland": ["Dublin", "Cork", "Galway"],
}

# Definimos los canales de adquisición disponibles.
acquisition_channels = [
    "organic",
    "paid_ads",
    "social_media",
    "email",
    "referral",
]

In [10]:
# Generamos los clientes sintéticos.
customers_data = []

for customer_id in range(1, NUM_CUSTOMERS + 1):
    country = fake.random_element(elements=list(locations.keys()))
    city = fake.random_element(elements=locations[country])

    customers_data.append({
        "customer_id": customer_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "country": country,
        "city": city,
        "acquisition_channel": fake.random_element(
            elements=acquisition_channels
        ),
        "registration_date": fake.date_between(
            start_date="-3y",
            end_date="today"
        ),
    })

customers_df = pd.DataFrame(customers_data)

print(f"Clientes generados: {len(customers_df)}")
customers_df.head()

Clientes generados: 500


,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date
0,1,Nilda,Pazos,villarvidal@example.org,+34883 18 56 68,Netherlands,Amsterdam,email,2024-07-24
1,2,Félix,Arrieta,jimena07@example.com,+34902 29 21 26,France,Marseille,referral,2025-08-16
2,3,Curro,Cornejo,pratsale@example.org,+34 902 75 93 90,Belgium,Antwerp,referral,2024-07-24
3,4,Albina,Gómez,tsilva@example.net,+34 925 15 61 62,Portugal,Porto,organic,2025-09-11
4,5,Amaro,Sebastián,ferreronazaret@example.com,+34973788653,Spain,Barcelona,email,2024-06-28


In [11]:
# Validamos la cantidad de clientes y la unicidad de sus IDs y emails.
print("Número de clientes:", len(customers_df))
print("IDs únicos:", customers_df["customer_id"].is_unique)
print("Emails únicos:", customers_df["email"].is_unique)
print("Valores nulos:", customers_df.isnull().sum().sum())

Número de clientes: 500
IDs únicos: True
Emails únicos: True
Valores nulos: 0


In [12]:
# Definimos algunos nombres base para cada categoría.
product_names = {
    1: ["Galaxy", "Pixel", "iPhone", "Xperia", "Redmi"],
    2: ["ThinkPad", "MacBook", "Inspiron", "ZenBook", "IdeaPad"],
    3: ["AirPods", "WH", "QuietComfort", "SoundLink", "Momentum"],
    4: ["MX Master", "K120", "Brio", "Ergo", "Mechanical"],
    5: ["Apple Watch", "Galaxy Watch", "Fitbit", "Garmin", "Amazfit"],
    6: ["USB-C Hub", "Charger", "Power Bank", "Cable", "Stand"],
    7: ["PlayStation", "Xbox", "Razer", "Steam Deck", "Gaming"],
}

In [15]:
# Generamos los productos sintéticos.
products_data = []

for product_id in range(1, NUM_PRODUCTS + 1):
    category_id = fake.random_int(
        min=1,
        max=len(categories_df)
    )

    base_name = fake.random_element(
        elements=product_names[category_id]
    )

    product_name = f"{base_name} {fake.bothify(text='???-###')}"

    # Generamos un precio de venta entre 20 y 1500 euros.
    price = round(
        fake.random.uniform(20, 1500),
        2
    )

    # Generamos un coste inferior al precio de venta.
    cost = round(
        price * fake.random.uniform(0.55, 0.80),
        2
    )

    products_data.append({
        "product_id": product_id,
        "category_id": category_id,
        "name": product_name,
        "description": fake.sentence(nb_words=10),
        "price": price,
        "cost": cost,
        "stock": fake.random_int(min=0, max=500),
        "is_active": fake.boolean(chance_of_getting_true=90),
    })

products_df = pd.DataFrame(products_data)

print(f"Productos generados: {len(products_df)}")
products_df.head()

Productos generados: 70


,product_id,category_id,name,description,price,cost,stock,is_active
0,1,3,QuietComfort WCA-670,Fecha sociedad cuadro análisis niños ir san es...,1441.58,942.42,141,True
1,2,4,Mechanical OkD-556,Régimen único mayor radio congreso quien socia...,1022.81,654.38,231,True
2,3,4,Mechanical hpw-405,Nada ciudad amor única existencia cuanto.,843.56,514.25,386,True
3,4,5,Amazfit kYh-783,Ella maría elecciones última principio salud c...,745.63,559.62,328,False
4,5,2,Inspiron qSH-391,Mucho rey corazón señora enero libre chile señ...,1101.75,790.92,185,True


In [16]:
# Validamos los productos generados.
print("Número de productos:", len(products_df))
print("IDs únicos:", products_df["product_id"].is_unique)
print("Valores nulos:", products_df.isnull().sum().sum())
print("Coste siempre menor que precio:", (products_df["cost"] < products_df["price"]).all())

Número de productos: 70
IDs únicos: True
Valores nulos: 0
Coste siempre menor que precio: True


In [17]:
# Comprobamos que todos los productos pertenecen a una categoría existente.
categorias_validas = set(categories_df["category_id"])

productos_sin_categoria = products_df[
    ~products_df["category_id"].isin(categorias_validas)
]

print(
    "Productos con category_id inválido:",
    len(productos_sin_categoria)
)

Productos con category_id inválido: 0


In [18]:
# Definimos los estados posibles de un pedido.
order_statuses = [
    "pending",
    "confirmed",
    "shipped",
    "delivered",
    "cancelled",
    "returned",
]

In [19]:
# Generamos los pedidos sintéticos.
orders_data = []

for order_id in range(1, NUM_ORDERS + 1):

    # Seleccionamos un cliente existente.
    customer = customers_df.sample(n=1).iloc[0]

    # La fecha del pedido siempre es posterior al registro del cliente.
    registration_date = customer["registration_date"]

    order_date = fake.date_between(
        start_date=registration_date,
        end_date="today"
    )

    status = fake.random_element(elements=order_statuses)

    # Inicializamos las fechas opcionales.
    shipped_date = None
    delivered_date = None

    # Los pedidos enviados pueden tener fecha de envío.
    if status in ["shipped", "delivered", "returned"]:
        shipped_date = fake.date_between(
            start_date=order_date,
            end_date="today"
        )

    # Los pedidos entregados o devueltos deben haber sido enviados.
    if status in ["delivered", "returned"]:
        delivered_date = fake.date_between(
            start_date=shipped_date,
            end_date="today"
        )

    country = customer["country"]
    city = customer["city"]

    orders_data.append({
        "order_id": order_id,
        "customer_id": customer["customer_id"],
        "status": status,
        "shipping_address": fake.address().replace("\n", ", "),
        "shipping_city": city,
        "shipping_country": country,
        "order_date": order_date,
        "shipped_date": shipped_date,
        "delivered_date": delivered_date,
    })

orders_df = pd.DataFrame(orders_data)

print(f"Pedidos generados: {len(orders_df)}")
orders_df.head()

Pedidos generados: 2000


,order_id,customer_id,status,shipping_address,shipping_city,shipping_country,order_date,shipped_date,delivered_date
0,1,486,confirmed,"Cañada de Macarena Orozco 1 Apt. 49 , Ávila, 0...",Lyon,France,2026-06-11,None,None
1,2,53,cancelled,"Plaza Elba Bustos 35, Alicante, 08900",Ghent,Belgium,2025-11-14,None,None
2,3,130,returned,"Urbanización de Estrella Martin 30 Piso 5 , Al...",Galway,Ireland,2025-12-01,2026-04-15,2026-06-16
3,4,461,returned,"Ronda de Olga Cañete 42 Puerta 9 , Cantabria, ...",Dublin,Ireland,2026-03-26,2026-07-10,2026-08-16
4,5,133,confirmed,"Plaza de Águeda Molins 44 Puerta 3 , Huesca, 3...",Bilbao,Spain,2026-02-22,None,None


In [20]:
# Validamos la cantidad y la unicidad de los pedidos.
print("Número de pedidos:", len(orders_df))
print("IDs únicos:", orders_df["order_id"].is_unique)
print("Valores nulos:", orders_df.isnull().sum().sum())

Número de pedidos: 2000
IDs únicos: True
Valores nulos: 2296


In [21]:
# Comprobamos que todos los pedidos pertenecen a clientes existentes.
clientes_validos = set(customers_df["customer_id"])

pedidos_sin_cliente = orders_df[
    ~orders_df["customer_id"].isin(clientes_validos)
]

print(
    "Pedidos con customer_id inválido:",
    len(pedidos_sin_cliente)
)

Pedidos con customer_id inválido: 0


In [22]:
# Comprobamos que las fechas de los pedidos siguen un orden lógico.
envios_invalidos = orders_df[
    orders_df["shipped_date"].notna()
    & (orders_df["shipped_date"] < orders_df["order_date"])
]

entregas_invalidas = orders_df[
    orders_df["delivered_date"].notna()
    & (orders_df["delivered_date"] < orders_df["shipped_date"])
]

print("Envíos anteriores al pedido:", len(envios_invalidos))
print("Entregas anteriores al envío:", len(entregas_invalidas))

Envíos anteriores al pedido: 0
Entregas anteriores al envío: 0


In [23]:
# Generamos las líneas de pedido.
order_items_data = []

order_item_id = 1

for _, order in orders_df.iterrows():

    # Cada pedido tendrá entre 1 y 4 productos.
    num_items = fake.random_int(min=1, max=4)

    # Seleccionamos productos diferentes para cada pedido.
    selected_products = products_df.sample(
        n=num_items,
        replace=False
    )

    for _, product in selected_products.iterrows():

        # Generamos la cantidad comprada.
        quantity = fake.random_int(min=1, max=3)

        # El precio histórico puede ser inferior al precio actual.
        unit_price = round(
            product["price"] * fake.random.uniform(0.85, 1.00),
            2
        )

        # Generamos un descuento entre 0% y 15%.
        discount = round(
            fake.random.uniform(0.00, 0.15),
            2
        )

        order_items_data.append({
            "order_item_id": order_item_id,
            "order_id": order["order_id"],
            "product_id": product["product_id"],
            "quantity": quantity,
            "unit_price": unit_price,
            "discount": discount,
        })

        order_item_id += 1

order_items_df = pd.DataFrame(order_items_data)

print(f"Líneas de pedido generadas: {len(order_items_df)}")
order_items_df.head()

Líneas de pedido generadas: 5008


,order_item_id,order_id,product_id,quantity,unit_price,discount
0,1,1,50,1,896.32,0.12
1,2,1,40,3,147.71,0.14
2,3,1,29,2,604.72,0.15
3,4,1,18,3,415.64,0.02
4,5,2,52,3,1281.85,0.00


In [24]:
# Comprobamos la cantidad de líneas y la media de productos por pedido.
print("Número de líneas:", len(order_items_df))

media_items = (
    len(order_items_df) / len(orders_df)
)

print("Media de líneas por pedido:", round(media_items, 2))

Número de líneas: 5008
Media de líneas por pedido: 2.5


In [25]:
# Comprobamos que todas las líneas pertenecen a pedidos existentes.
pedidos_validos = set(orders_df["order_id"])

lineas_sin_pedido = order_items_df[
    ~order_items_df["order_id"].isin(pedidos_validos)
]

print(
    "Líneas con order_id inválido:",
    len(lineas_sin_pedido)
)

Líneas con order_id inválido: 0


In [26]:
# Comprobamos que todas las líneas utilizan productos existentes.
productos_validos = set(products_df["product_id"])

lineas_sin_producto = order_items_df[
    ~order_items_df["product_id"].isin(productos_validos)
]

print(
    "Líneas con product_id inválido:",
    len(lineas_sin_producto)
)

Líneas con product_id inválido: 0


In [27]:
# Comprobamos que las cantidades y precios sean válidos.
print(
    "Cantidades mayores que 0:",
    (order_items_df["quantity"] > 0).all()
)

print(
    "Precios mayores que 0:",
    (order_items_df["unit_price"] > 0).all()
)

print(
    "Descuentos entre 0 y 1:",
    order_items_df["discount"].between(0, 1).all()
)

Cantidades mayores que 0: True
Precios mayores que 0: True
Descuentos entre 0 y 1: True


In [28]:
# Comprobamos que todos los pedidos tienen al menos una línea.
pedidos_con_items = order_items_df["order_id"].nunique()

print("Pedidos con al menos una línea:", pedidos_con_items)
print("Total de pedidos:", len(orders_df))

Pedidos con al menos una línea: 2000
Total de pedidos: 2000


In [30]:
# Definimos los métodos y estados posibles de los pagos.
payment_methods = [
    "card",
    "paypal",
    "bank_transfer",
    "apple_pay",
    "google_pay",
]

payment_statuses = [
    "completed",
    "refunded",
    "pending",
    "failed",
]

In [31]:
# Generamos un pago para cada pedido.
payments_data = []

for _, order in orders_df.iterrows():

    order_id = order["order_id"]

    # Obtenemos las líneas correspondientes al pedido.
    items = order_items_df[
        order_items_df["order_id"] == order_id
    ]

    # Calculamos el importe total del pedido.
    subtotal = (
        items["quantity"] * items["unit_price"]
    ).sum()

    descuentos = (
        items["quantity"]
        * items["unit_price"]
        * items["discount"]
    ).sum()

    amount = round(subtotal - descuentos, 2)

    # Seleccionamos el estado del pago.
    payment_status = fake.random_element(
        elements=payment_statuses
    )

    # Los pagos pendientes o fallidos pueden no haberse completado.
    # Aun así, registramos el importe asociado al intento de pago.
    payment_date = order["order_date"]

    payments_data.append({
        "payment_id": order_id,
        "order_id": order_id,
        "payment_method": fake.random_element(
            elements=payment_methods
        ),
        "payment_status": payment_status,
        "amount": amount,
        "payment_date": payment_date,
    })

payments_df = pd.DataFrame(payments_data)

print(f"Pagos generados: {len(payments_df)}")
payments_df.head()

Pagos generados: 2000


,payment_id,order_id,payment_method,payment_status,amount,payment_date
0,1,1,apple_pay,pending,3419.86,2026-06-11
1,2,2,card,completed,12769.25,2025-11-14
2,3,3,bank_transfer,failed,174.24,2025-12-01
3,4,4,bank_transfer,failed,6791.48,2026-03-26
4,5,5,card,pending,5501.67,2026-02-22


In [32]:
# Validamos la cantidad y unicidad de los pagos.
print("Número de pagos:", len(payments_df))
print("IDs únicos:", payments_df["payment_id"].is_unique)
print("Valores nulos:", payments_df.isnull().sum().sum())

Número de pagos: 2000
IDs únicos: True
Valores nulos: 0


In [33]:
# Comprobamos que todos los pagos pertenecen a pedidos existentes.
pedidos_validos = set(orders_df["order_id"])

pagos_sin_pedido = payments_df[
    ~payments_df["order_id"].isin(pedidos_validos)
]

print(
    "Pagos con order_id inválido:",
    len(pagos_sin_pedido)
)

Pagos con order_id inválido: 0


In [34]:
# Comprobamos que todos los importes sean válidos.
print(
    "Importes mayores o iguales a 0:",
    (payments_df["amount"] >= 0).all()
)

print(
    "Importes nulos:",
    payments_df["amount"].isnull().sum()
)

Importes mayores o iguales a 0: True
Importes nulos: 0


In [35]:
# Comprobamos que cada pedido tiene exactamente un pago.
pagos_por_pedido = payments_df.groupby("order_id").size()

print(
    "Pedidos con exactamente un pago:",
    (pagos_por_pedido == 1).sum()
)

print(
    "Total de pedidos:",
    len(orders_df)
)

Pedidos con exactamente un pago: 2000
Total de pedidos: 2000


In [36]:
# Definimos los valores posibles para las valoraciones.
ratings = [1, 2, 3, 4, 5]

# Definimos algunos comentarios para generar opiniones realistas.
review_comments = [
    "Muy buen producto, cumple con lo esperado.",
    "Buena relación calidad-precio.",
    "La calidad es excelente.",
    "Funciona correctamente y llegó en buen estado.",
    "Estoy satisfecho con la compra.",
    "El producto podría mejorar.",
    "No cumplió completamente mis expectativas.",
    "Muy recomendable.",
]

In [37]:
# Seleccionamos las líneas pertenecientes a pedidos entregados.
delivered_orders = orders_df[
    orders_df["status"] == "delivered"
]

delivered_order_items = order_items_df[
    order_items_df["order_id"].isin(
        delivered_orders["order_id"]
    )
].copy()

print(
    "Pedidos entregados:",
    len(delivered_orders)
)

print(
    "Líneas de pedidos entregados:",
    len(delivered_order_items)
)

Pedidos entregados: 352
Líneas de pedidos entregados: 876


In [38]:
# Calculamos aproximadamente el 35 % de las líneas entregadas.
num_reviews = int(
    len(delivered_order_items) * 0.35
)

# Seleccionamos líneas diferentes para recibir una valoración.
review_items = delivered_order_items.sample(
    n=num_reviews,
    random_state=42
)

print("Reviews a generar:", num_reviews)

Reviews a generar: 306


In [39]:
# Generamos las valoraciones sintéticas.
reviews_data = []

for review_id, (_, item) in enumerate(
    review_items.iterrows(),
    start=1
):
    order = orders_df[
        orders_df["order_id"] == item["order_id"]
    ].iloc[0]

    review_date = fake.date_between(
        start_date=order["delivered_date"],
        end_date="today"
    )

    reviews_data.append({
        "review_id": review_id,
        "order_item_id": item["order_item_id"],
        "rating": fake.random_element(
            elements=ratings
        ),
        "comment": fake.random_element(
            elements=review_comments
        ),
        "review_date": review_date,
    })

reviews_df = pd.DataFrame(reviews_data)

print(f"Reviews generadas: {len(reviews_df)}")
reviews_df.head()

Reviews generadas: 306


,review_id,order_item_id,rating,comment,review_date
0,1,2073.0,5,El producto podría mejorar.,2026-08-29
1,2,3799.0,4,Muy recomendable.,2026-08-30
2,3,4788.0,4,No cumplió completamente mis expectativas.,2026-06-21
3,4,4099.0,1,El producto podría mejorar.,2026-08-22
4,5,1059.0,4,Estoy satisfecho con la compra.,2026-04-23


In [40]:
# Validamos la cantidad y unicidad de las reviews.
print("Número de reviews:", len(reviews_df))
print("IDs únicos:", reviews_df["review_id"].is_unique)
print("Valores nulos:", reviews_df.isnull().sum().sum())

Número de reviews: 306
IDs únicos: True
Valores nulos: 0


In [41]:
# Comprobamos que todas las reviews pertenecen a líneas existentes.
items_validos = set(
    order_items_df["order_item_id"]
)

reviews_sin_item = reviews_df[
    ~reviews_df["order_item_id"].isin(items_validos)
]

print(
    "Reviews con order_item_id inválido:",
    len(reviews_sin_item)
)

Reviews con order_item_id inválido: 0


In [42]:
# Comprobamos que todas las reviews pertenecen a pedidos entregados.
review_items_check = order_items_df[
    order_items_df["order_item_id"].isin(
        reviews_df["order_item_id"]
    )
]

reviews_no_entregadas = review_items_check[
    ~review_items_check["order_id"].isin(
        delivered_orders["order_id"]
    )
]

print(
    "Reviews sobre pedidos no entregados:",
    len(reviews_no_entregadas)
)

Reviews sobre pedidos no entregados: 0


In [43]:
# Calculamos el porcentaje de líneas entregadas que tienen review.
porcentaje_reviews = (
    len(reviews_df)
    / len(delivered_order_items)
) * 100

print(
    "Porcentaje de líneas entregadas con review:",
    round(porcentaje_reviews, 2),
    "%"
)

Porcentaje de líneas entregadas con review: 34.93 %
